# Feature Engineering

This notebook creates the input features used by the predictive model.

These features are designed for forecasting rather than for descriptive EDA.
In particular, each feature row should use only information that would have been available before the target draw, so the current draw does not leak directly into the feature values.

### 1. Library Imports

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
APP_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "src" / "notebook_support.py").exists() and (candidate / "data").exists():
        APP_ROOT = candidate
        break
    if (candidate / "app" / "src" / "notebook_support.py").exists() and (candidate / "app" / "data").exists():
        APP_ROOT = candidate / "app"
        break

if APP_ROOT is None:
    raise RuntimeError(f"Could not locate app root from {cwd}")

if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

from src.notebook_support import describe_notebook_environment
describe_notebook_environment(APP_ROOT)

import pandas as pd

from src.config import FEATURE_LOTTO_FILE
from src.features import build_feature_dataset, build_model_feature_bundle, describe_feature_sets


### 2. Build the Feature Set

The current feature set is built from two components:

- Number-specific appearance frequency over the most recent `window` draws
- Gap features measuring how many draws have passed since each number last appeared

The resulting dataset will be used as the model input in the baseline notebook.

In [2]:
WINDOW = 20
base_feature_df = build_feature_dataset(window=WINDOW, save=True)
base_feature_df.head()


,freq_1,freq_2,freq_3,freq_4,freq_5,freq_6,freq_7,freq_8,freq_9,freq_10,...,gap_36,gap_37,gap_38,gap_39,gap_40,gap_41,gap_42,gap_43,gap_44,gap_45
0,1.0,4.0,3.0,3.0,0.0,3.0,2.0,1.0,4.0,2.0,...,10.0,4.0,2.0,2.0,2.0,10.0,8.0,2.0,11.0,9.0
1,1.0,4.0,3.0,3.0,0.0,4.0,2.0,1.0,4.0,1.0,...,11.0,5.0,3.0,3.0,3.0,11.0,9.0,3.0,12.0,10.0
2,1.0,4.0,3.0,4.0,1.0,5.0,2.0,2.0,3.0,1.0,...,12.0,6.0,4.0,1.0,4.0,12.0,10.0,4.0,13.0,11.0
3,1.0,4.0,3.0,4.0,2.0,5.0,2.0,2.0,3.0,1.0,...,13.0,7.0,5.0,2.0,5.0,13.0,1.0,5.0,14.0,12.0
4,1.0,4.0,3.0,4.0,2.0,5.0,3.0,3.0,3.0,1.0,...,1.0,8.0,6.0,3.0,6.0,14.0,2.0,1.0,15.0,13.0


In [3]:
base_feature_df.shape


(1202, 90)

In [4]:
feature_bundle = build_model_feature_bundle(window=WINDOW, save_base=False)
describe_feature_sets(feature_bundle)


,feature_set,n_features,description
0,base,90,Rolling frequency and gap features only.
1,base_plus_pattern,118,Base features plus draw-level internal pattern...
2,base_plus_context,121,Base features plus calendar and weather context.
3,full_feature_set,149,"Base, internal pattern, and context features c..."


### 3. Summary

This notebook builds a forecasting-oriented feature table from the cleaned lotto history.

The final dataset combines rolling frequency features and gap features, and each row is structured so that it uses only information available before the target draw. This feature table is the direct input to the baseline model notebook.